# 60 Machine Learning Algorithms

40 scikit-learn + 10 PyTorch + 10 TensorFlow/Keras

## Before you start

- Run the notebook from top to bottom.
- The examples use small synthetic datasets so they can run without downloading data.
- Synthetic results are for learning the workflow, not for judging which model is best in a real problem.
- TensorFlow/Keras is optional. Install it with `pip install tensorflow` to run that section.

Required packages: `numpy`, `pandas`, `scikit-learn`, and `torch`. Install them with `pip install numpy pandas scikit-learn torch` if needed.

## 1. Setup data dan helper scikit-learn

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_regression, make_classification, make_blobs
from sklearn.model_selection import (
    train_test_split, cross_validate, StratifiedKFold, KFold
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, Binarizer, KBinsDiscretizer, OneHotEncoder
)
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)

In [4]:
RANDOM_STATE = 1
np.random.seed(RANDOM_STATE)

In [6]:
# Dataset regresi
X_reg, y_reg = make_regression(
    n_samples=400, n_features=10, noise=12, random_state=RANDOM_STATE
)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_STATE
)

# Dataset classification
X_clf, y_clf = make_classification(
    n_samples=400, n_features=12, n_informative=8, n_redundant=2,
    n_classes=2, random_state=RANDOM_STATE
)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.25, stratify=y_clf, random_state=RANDOM_STATE
)

# Dataset clustering
X_cluster, _ = make_blobs(
    n_samples=300, centers=4, n_features=2, cluster_std=1.2,
    random_state=RANDOM_STATE
)
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

# Dataset anomaly detection:
rng = np.random.default_rng(RANDOM_STATE)
X_normal = rng.normal(0, 1, size=(280, 5))
X_anomaly = rng.normal(5, 1, size=(20, 5))
X_outlier = np.vstack([X_normal, X_anomaly])
y_outlier = np.r_[np.zeros(len(X_normal), dtype=int), np.ones(len(X_anomaly), dtype=int)]

# Helper evaluasi. Scaling ada di dalam pipeline supaya tidak leakage.
def evaluate_classifier(estimator, scale=False, name=None):
    model = make_pipeline(StandardScaler(), estimator) if scale else estimator
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    print(name or estimator.__class__.__name__)
    print(f"Accuracy: {accuracy_score(yc_test, pred):.3f} | F1: {f1_score(yc_test, pred):.3f}")
    return model

def evaluate_regressor(estimator, scale=False, name=None):
    model = make_pipeline(StandardScaler(), estimator) if scale else estimator
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    rmse = np.sqrt(mean_squared_error(yr_test, pred))
    print(name or estimator.__class__.__name__)
    print(f"R²: {r2_score(yr_test, pred):.3f} | RMSE: {rmse:.3f}")
    return model

def evaluate_cluster(estimator, name=None):
    labels = estimator.fit_predict(X_cluster_scaled)
    n_labels = len(set(labels)) - (1 if -1 in labels else 0)
    valid = labels != -1
    score = silhouette_score(X_cluster_scaled[valid], labels[valid]) if n_labels > 1 and valid.sum() > n_labels else float('nan')
    noise_count = int(np.sum(labels == -1))
    print(name or estimator.__class__.__name__)
    print(f"Jumlah cluster: {n_labels} | Noise: {noise_count} | Silhouette: {score:.3f}")
    return estimator, labels


## 2. scikit-learn 

### A. Linear models dan model dasar

#### 01. Linear Regression

In [7]:
from sklearn.linear_model import LinearRegression
model = evaluate_regressor(LinearRegression())

LinearRegression
R²: 0.995 | RMSE: 12.891


#### 02. Ridge Regression

In [8]:
from sklearn.linear_model import Ridge
model = evaluate_regressor(Ridge(alpha=1.0), scale=True)

Ridge
R²: 0.995 | RMSE: 12.980


#### 03. Lasso Regression

In [9]:
from sklearn.linear_model import Lasso
model = evaluate_regressor(Lasso(alpha=0.1), scale=True)

Lasso
R²: 0.995 | RMSE: 12.927


#### 04. Elastic Net

In [10]:
from sklearn.linear_model import ElasticNet
model = evaluate_regressor(ElasticNet(alpha=0.1, l1_ratio=0.5), scale=True)

ElasticNet
R²: 0.992 | RMSE: 16.680


#### 05. Logistic Regression

In [11]:
from sklearn.linear_model import LogisticRegression
model = evaluate_classifier(LogisticRegression(max_iter=1000), scale=True)

LogisticRegression
Accuracy: 0.820 | F1: 0.820


#### 06. SGD Classifier

In [12]:
from sklearn.linear_model import SGDClassifier
model = evaluate_classifier(SGDClassifier(max_iter=2000, tol=1e-3, random_state=RANDOM_STATE), scale=True)

SGDClassifier
Accuracy: 0.690 | F1: 0.674


#### 07. Perceptron

In [13]:
from sklearn.linear_model import Perceptron
model = evaluate_classifier(Perceptron(max_iter=2000, random_state=RANDOM_STATE), scale=True)

Perceptron
Accuracy: 0.750 | F1: 0.737


#### 08. Passive Aggressive Classifier

In [14]:
from sklearn.linear_model import PassiveAggressiveClassifier
model = evaluate_classifier(PassiveAggressiveClassifier(max_iter=2000, random_state=RANDOM_STATE), scale=True)

PassiveAggressiveClassifier
Accuracy: 0.810 | F1: 0.800


C:\Users\FARUQ\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


### B. Nearest neighbors dan Support Vector Machine

#### 09. K-Nearest Neighbors Classifier

In [15]:
from sklearn.neighbors import KNeighborsClassifier
model = evaluate_classifier(KNeighborsClassifier(n_neighbors=5), scale=True)

KNeighborsClassifier
Accuracy: 0.930 | F1: 0.929


#### 10. K-Nearest Neighbors Regressor

In [16]:
from sklearn.neighbors import KNeighborsRegressor
model = evaluate_regressor(KNeighborsRegressor(n_neighbors=5), scale=True)

KNeighborsRegressor
R²: 0.711 | RMSE: 102.099


#### 11. Support Vector Classifier (SVC)

In [17]:
from sklearn.svm import SVC
model = evaluate_classifier(SVC(kernel="rbf", C=1.0), scale=True)

SVC
Accuracy: 0.960 | F1: 0.961


#### 12. Support Vector Regression (SVR)

In [18]:
from sklearn.svm import SVR
model = evaluate_regressor(SVR(kernel="rbf", C=10.0, epsilon=0.1), scale=True)

SVR
R²: 0.540 | RMSE: 128.953


### C. Decision tree dan ensemble learning

#### 13. Decision Tree Classifier

In [19]:
from sklearn.tree import DecisionTreeClassifier
model = evaluate_classifier(DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))

DecisionTreeClassifier
Accuracy: 0.860 | F1: 0.854


#### 14. Decision Tree Regressor

In [20]:
from sklearn.tree import DecisionTreeRegressor
model = evaluate_regressor(DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE))

DecisionTreeRegressor
R²: 0.259 | RMSE: 163.652


#### 15. Random Forest Classifier

In [21]:
from sklearn.ensemble import RandomForestClassifier
model = evaluate_classifier(RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1))

RandomForestClassifier
Accuracy: 0.870 | F1: 0.869


#### 16. Random Forest Regressor

In [22]:
from sklearn.ensemble import RandomForestRegressor
model = evaluate_regressor(RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1))

RandomForestRegressor
R²: 0.714 | RMSE: 101.721


#### 17. Extra Trees Classifier

In [23]:
from sklearn.ensemble import ExtraTreesClassifier
model = evaluate_classifier(ExtraTreesClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1))

ExtraTreesClassifier
Accuracy: 0.900 | F1: 0.898


#### 18. Extra Trees Regressor

In [24]:
from sklearn.ensemble import ExtraTreesRegressor
model = evaluate_regressor(ExtraTreesRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1))

ExtraTreesRegressor
R²: 0.792 | RMSE: 86.749


#### 19. Gradient Boosting Classifier

In [25]:
from sklearn.ensemble import GradientBoostingClassifier
model = evaluate_classifier(GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=RANDOM_STATE))

GradientBoostingClassifier
Accuracy: 0.890 | F1: 0.887


#### 20. Gradient Boosting Regressor

In [26]:
from sklearn.ensemble import GradientBoostingRegressor
model = evaluate_regressor(GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, random_state=RANDOM_STATE))

GradientBoostingRegressor
R²: 0.743 | RMSE: 96.440


#### 21. Histogram Gradient Boosting Classifier

In [27]:
from sklearn.ensemble import HistGradientBoostingClassifier
model = evaluate_classifier(HistGradientBoostingClassifier(max_iter=100, random_state=RANDOM_STATE))

HistGradientBoostingClassifier
Accuracy: 0.910 | F1: 0.909


#### 22. Histogram Gradient Boosting Regressor

In [28]:
from sklearn.ensemble import HistGradientBoostingRegressor
model = evaluate_regressor(HistGradientBoostingRegressor(max_iter=100, random_state=RANDOM_STATE))

HistGradientBoostingRegressor
R²: 0.836 | RMSE: 76.962


#### 23. AdaBoost Classifier

In [29]:
from sklearn.ensemble import AdaBoostClassifier
model = evaluate_classifier(AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE))

AdaBoostClassifier
Accuracy: 0.800 | F1: 0.800


#### 24. Bagging Classifier

In [30]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
base_tree = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
model = evaluate_classifier(BaggingClassifier(estimator=base_tree, n_estimators=30, random_state=RANDOM_STATE, n_jobs=-1))

BaggingClassifier
Accuracy: 0.820 | F1: 0.816


#### 25. Voting Classifier

In [31]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
voting = VotingClassifier(
    estimators=[
        ("lr", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
        ("extra", ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_STATE)),
    ], voting="soft"
)
model = evaluate_classifier(voting)

VotingClassifier
Accuracy: 0.870 | F1: 0.869


#### 26. Stacking Classifier

In [32]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
stacking = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=80, random_state=RANDOM_STATE)),
        ("svc", make_pipeline(StandardScaler(), SVC(probability=True, random_state=RANDOM_STATE))),
    ], final_estimator=LogisticRegression(max_iter=1000), cv=3
)
model = evaluate_classifier(stacking)

C:\Users\FARUQ\anaconda3\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


StackingClassifier
Accuracy: 0.930 | F1: 0.931


C:\Users\FARUQ\anaconda3\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\FARUQ\anaconda3\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\FARUQ\anaconda3\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


### D. Probabilistic dan discriminant models

#### 27. Gaussian Naive Bayes

In [33]:
from sklearn.naive_bayes import GaussianNB
model = evaluate_classifier(GaussianNB())

GaussianNB
Accuracy: 0.800 | F1: 0.800


#### 28. Multinomial Naive Bayes

In [34]:
from sklearn.naive_bayes import MultinomialNB

# MultinomialNB menerima fitur non-negative. Untuk contoh ini, fitur diskalakan ke 0-1.
model = make_pipeline(MinMaxScaler(), MultinomialNB(alpha=1.0))
model.fit(Xc_train, yc_train)
pred = model.predict(Xc_test)
print("Accuracy:", round(accuracy_score(yc_test, pred), 3))

Accuracy: 0.7


#### 29. Bernoulli Naive Bayes

In [35]:
from sklearn.naive_bayes import BernoulliNB

# BernoulliNB bekerja dengan fitur biner (0 atau 1), bukan sekadar fitur yang diskalakan.
model = make_pipeline(
    KBinsDiscretizer(n_bins=2, encode="onehot-dense", strategy="quantile"),
    BernoulliNB(alpha=1.0)
)
model.fit(Xc_train, yc_train)
pred = model.predict(Xc_test)
print("Accuracy:", round(accuracy_score(yc_test, pred), 3))

Accuracy: 0.67


#### 30. Linear Discriminant Analysis (LDA)

In [36]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
model = evaluate_classifier(LinearDiscriminantAnalysis(), scale=True)

LinearDiscriminantAnalysis
Accuracy: 0.820 | F1: 0.820


#### 31. Quadratic Discriminant Analysis (QDA)

In [37]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
model = evaluate_classifier(QuadraticDiscriminantAnalysis(reg_param=0.1))

QuadraticDiscriminantAnalysis
Accuracy: 0.910 | F1: 0.913


### E. Clustering

#### 32. K-Means

In [38]:
from sklearn.cluster import KMeans
model, labels = evaluate_cluster(KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE))

KMeans
Jumlah cluster: 4 | Noise: 0 | Silhouette: 0.569


#### 33. Mini-Batch K-Means

In [39]:
from sklearn.cluster import MiniBatchKMeans
model, labels = evaluate_cluster(MiniBatchKMeans(n_clusters=4, batch_size=64, n_init=10, random_state=RANDOM_STATE))

MiniBatchKMeans
Jumlah cluster: 4 | Noise: 0 | Silhouette: 0.569


#### 34. DBSCAN

In [40]:
from sklearn.cluster import DBSCAN
model, labels = evaluate_cluster(DBSCAN(eps=0.35, min_samples=5))

DBSCAN
Jumlah cluster: 2 | Noise: 4 | Silhouette: 0.683


#### 35. Agglomerative Clustering

In [41]:
from sklearn.cluster import AgglomerativeClustering
model, labels = evaluate_cluster(AgglomerativeClustering(n_clusters=4, linkage="ward"))

AgglomerativeClustering
Jumlah cluster: 4 | Noise: 0 | Silhouette: 0.568


#### 36. Spectral Clustering

In [42]:
from sklearn.cluster import SpectralClustering
model, labels = evaluate_cluster(SpectralClustering(n_clusters=4, affinity="nearest_neighbors", random_state=RANDOM_STATE))

C:\Users\FARUQ\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


SpectralClustering
Jumlah cluster: 4 | Noise: 0 | Silhouette: 0.568


#### 37. Gaussian Mixture Model (GMM)

In [43]:
from sklearn.mixture import GaussianMixture

model = GaussianMixture(n_components=4, random_state=RANDOM_STATE)
labels = model.fit_predict(X_cluster_scaled)
print("Banyak cluster:", len(np.unique(labels)))
print("AIC:", round(model.aic(X_cluster_scaled), 2))
print("BIC:", round(model.bic(X_cluster_scaled), 2))
print("Contoh probabilitas keanggotaan:\n", model.predict_proba(X_cluster_scaled[:3]))

Banyak cluster: 4
AIC: 1087.01
BIC: 1172.19
Contoh probabilitas keanggotaan:
 [[4.32811899e-18 1.00000000e+00 1.22318065e-24 1.87400674e-31]
 [3.37082541e-07 1.52361265e-41 9.99955078e-01 4.45852352e-05]
 [2.44352971e-16 1.00000000e+00 2.01369270e-23 4.93157358e-29]]


### F. Dimensionality reduction dan anomaly detection

#### 38. Principal Component Analysis (PCA)

In [44]:
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clf)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Shape sebelum:", X_clf.shape, "| sesudah PCA:", X_pca.shape)
print("Explained variance:", np.round(pca.explained_variance_ratio_, 3))
print("Total explained variance:", round(pca.explained_variance_ratio_.sum(), 3))

Shape sebelum: (400, 12) | sesudah PCA: (400, 2)
Explained variance: [0.242 0.15 ]
Total explained variance: 0.392


#### 39. Truncated SVD

In [45]:
from sklearn.decomposition import TruncatedSVD

# TruncatedSVD tidak melakukan centering seperti PCA.
svd = TruncatedSVD(n_components=5, random_state=RANDOM_STATE)
X_svd = svd.fit_transform(X_clf)
print("Shape hasil SVD:", X_svd.shape)
print("Total explained variance:", round(svd.explained_variance_ratio_.sum(), 3))

Shape hasil SVD: (400, 5)
Total explained variance: 0.833


#### 40. Isolation Forest

In [46]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(contamination=0.07, random_state=RANDOM_STATE)
pred = model.fit_predict(X_outlier)  # 1 = normal, -1 = anomaly
y_pred = (pred == -1).astype(int)

print("Jumlah anomaly terdeteksi:", int((pred == -1).sum()))
print("Precision:", round(precision_score(y_outlier, y_pred), 3))
print("Recall:", round(recall_score(y_outlier, y_pred), 3))
print("F1:", round(f1_score(y_outlier, y_pred), 3))

Jumlah anomaly terdeteksi: 21
Precision: 0.905
Recall: 0.95
F1: 0.927


## 3. PyTorch 

These examples use small, synthetic tensors. Run the following setup before selecting a model. `fit_torch` is a minimal training helper; for research purposes, incorporate a validation loop, scheduler, checkpointing, early stopping, and logging.

In [47]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Helper untuk membagi data menjadi train dan validation.
def make_torch_loaders(X, y, batch_size=32, val_size=0.2):
    dataset = TensorDataset(X, y)
    n_val = max(1, int(len(dataset) * val_size))
    n_train = len(dataset) - n_val
    train_set, val_set = random_split(
        dataset,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(RANDOM_STATE)
    )
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

# Tabular data
X_t = torch.randn(256, 12)
y_t = (X_t[:, :3].sum(dim=1) > 0).float().unsqueeze(1)
y_reg_t = (2 * X_t[:, 0] - X_t[:, 1] + 0.2 * torch.randn(256)).unsqueeze(1)
loader_bin, val_bin = make_torch_loaders(X_t, y_t)
loader_reg, val_reg = make_torch_loaders(X_t, y_reg_t)

def fit_torch(model, train_loader, val_loader, loss_fn, epochs=5, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_total = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_total += loss.item() * len(xb)

        model.eval()
        val_total = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                loss = loss_fn(model(xb), yb)
                val_total += loss.item() * len(xb)

        train_loss = train_total / len(train_loader.dataset)
        val_loss = val_total / len(val_loader.dataset)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch+1}/{epochs} - train: {train_loss:.4f} - val: {val_loss:.4f}")

    model.training_history = history
    return model

# Data sequence untuk RNN/LSTM/GRU/Transformer
X_seq = torch.randn(256, 20, 8)
y_seq = (X_seq.mean(dim=(1, 2)) > 0).float().unsqueeze(1)
loader_seq, val_seq = make_torch_loaders(X_seq, y_seq)


Device: cpu


### 01. MLP untuk klasifikasi tabular

In [48]:
class MLPClassifier(nn.Module):
    def __init__(self, n_features=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)  

model = fit_torch(MLPClassifier(), loader_bin, val_bin, nn.BCEWithLogitsLoss())

Epoch 1/5 - train: 0.6966 - val: 0.6814
Epoch 2/5 - train: 0.6738 - val: 0.6624
Epoch 3/5 - train: 0.6544 - val: 0.6410
Epoch 4/5 - train: 0.6385 - val: 0.6166
Epoch 5/5 - train: 0.6080 - val: 0.5893


### 02. MLP for Regression

In [49]:
class MLPRegressor(nn.Module):
    def __init__(self, n_features=12):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x): return self.net(x)

model = fit_torch(MLPRegressor(), loader_reg, val_reg, nn.MSELoss())

Epoch 1/5 - train: 4.7142 - val: 5.3542
Epoch 2/5 - train: 4.4947 - val: 5.1458
Epoch 3/5 - train: 4.2975 - val: 4.9532
Epoch 4/5 - train: 4.1013 - val: 4.7499
Epoch 5/5 - train: 3.9114 - val: 4.5378


### 03. CNN for Image Classification

In [50]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU(), nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# Dua pola sederhana: garis horizontal dan vertikal.
X_img = torch.zeros(256, 1, 28, 28)
y_img = torch.arange(256) % 2
for i in range(len(X_img)):
    if y_img[i] == 0:
        X_img[i, 0, 12:16, :] = 1.0
    else:
        X_img[i, 0, :, 12:16] = 1.0
X_img += 0.15 * torch.randn_like(X_img)

loader_img, val_img = make_torch_loaders(X_img, y_img)
model = fit_torch(SmallCNN(), loader_img, val_img, nn.CrossEntropyLoss())

Epoch 1/5 - train: 0.6008 - val: 0.3916
Epoch 2/5 - train: 0.2448 - val: 0.0622
Epoch 3/5 - train: 0.0274 - val: 0.0026
Epoch 4/5 - train: 0.0011 - val: 0.0002
Epoch 5/5 - train: 0.0001 - val: 0.0000


### 04. Simple RNN for Sequence Data

In [51]:
class RNNClassifier(nn.Module):
    def __init__(self, input_size=8, hidden_size=32):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.head(h_n[-1])

model = fit_torch(RNNClassifier(), loader_seq, val_seq, nn.BCEWithLogitsLoss())

Epoch 1/5 - train: 0.7198 - val: 0.7356
Epoch 2/5 - train: 0.7048 - val: 0.7251
Epoch 3/5 - train: 0.6928 - val: 0.7143
Epoch 4/5 - train: 0.6840 - val: 0.7052
Epoch 5/5 - train: 0.6738 - val: 0.6982


### 05. LSTM for Sequence Data

In [52]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=8, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1])

model = fit_torch(LSTMClassifier(), loader_seq, val_seq, nn.BCEWithLogitsLoss())

Epoch 1/5 - train: 0.6959 - val: 0.6950
Epoch 2/5 - train: 0.6912 - val: 0.6908
Epoch 3/5 - train: 0.6872 - val: 0.6866
Epoch 4/5 - train: 0.6833 - val: 0.6818
Epoch 5/5 - train: 0.6779 - val: 0.6763


### 06. GRU for Sequence Data

In [53]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size=8, hidden_size=32):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
    def forward(self, x):
        _, h_n = self.gru(x)
        return self.head(h_n[-1])

model = fit_torch(GRUClassifier(), loader_seq, val_seq, nn.BCEWithLogitsLoss())

Epoch 1/5 - train: 0.6941 - val: 0.6984
Epoch 2/5 - train: 0.6855 - val: 0.6897
Epoch 3/5 - train: 0.6780 - val: 0.6810
Epoch 4/5 - train: 0.6701 - val: 0.6722
Epoch 5/5 - train: 0.6628 - val: 0.6627


### 07. Autoencoder

In [54]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=12, latent_dim=3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 8), nn.ReLU(), nn.Linear(8, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 8), nn.ReLU(), nn.Linear(8, input_dim))
    def forward(self, x):
        return self.decoder(self.encoder(x))

loader_ae, val_ae = make_torch_loaders(X_t, X_t)
model = Autoencoder().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(5):
    model.train()
    train_loss = 0.0
    for (xb, _) in loader_ae:
        xb = xb.to(device)
        opt.zero_grad()
        reconstruction = model(xb)
        loss = loss_fn(reconstruction, xb)
        loss.backward()
        opt.step()
        train_loss += loss.item() * len(xb)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for (xb, _) in val_ae:
            xb = xb.to(device)
            val_loss += loss_fn(model(xb), xb).item() * len(xb)

    print(f"Epoch {epoch+1}/5 - train: {train_loss/len(loader_ae.dataset):.4f} - val: {val_loss/len(val_ae.dataset):.4f}")

Epoch 1/5 - train: 1.0654 - val: 1.2850
Epoch 2/5 - train: 1.0550 - val: 1.2750
Epoch 3/5 - train: 1.0454 - val: 1.2659
Epoch 4/5 - train: 1.0370 - val: 1.2579
Epoch 5/5 - train: 1.0295 - val: 1.2503


### 08. Transformer Encoder

In [55]:
class TransformerClassifier(nn.Module):
    def __init__(self, input_dim=8, d_model=32, nhead=4, n_classes=1, max_len=20):
        super().__init__()
        self.projection = nn.Linear(input_dim, d_model)
        self.position_embedding = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.position_embedding, mean=0.0, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        z = self.projection(x)
        z = z + self.position_embedding[:, :z.size(1), :]
        z = self.encoder(z)
        return self.head(z.mean(dim=1))

model = fit_torch(
    TransformerClassifier(), loader_seq, val_seq, nn.BCEWithLogitsLoss()
)

Epoch 1/5 - train: 0.7142 - val: 0.7875
Epoch 2/5 - train: 0.7038 - val: 0.6545
Epoch 3/5 - train: 0.6288 - val: 0.5839
Epoch 4/5 - train: 0.5691 - val: 0.5257
Epoch 5/5 - train: 0.5273 - val: 0.4568


### 09. Embedding + LSTM

In [56]:
class TokenLSTMClassifier(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=32, hidden_size=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 2)
    def forward(self, token_ids):
        embedded = self.embedding(token_ids)
        _, (h_n, _) = self.lstm(embedded)
        return self.head(h_n[-1])

X_tokens = torch.randint(1, 1000, (256, 30))
y_tokens = ((X_tokens >= 500).sum(dim=1) >= 15).long()
loader_tokens, val_tokens = make_torch_loaders(X_tokens, y_tokens)
model = fit_torch(
    TokenLSTMClassifier(), loader_tokens, val_tokens, nn.CrossEntropyLoss()
)

Epoch 1/5 - train: 0.6865 - val: 0.6828
Epoch 2/5 - train: 0.6780 - val: 0.6828
Epoch 3/5 - train: 0.6716 - val: 0.6827
Epoch 4/5 - train: 0.6653 - val: 0.6837
Epoch 5/5 - train: 0.6580 - val: 0.6853


### 10. GAN (Generator + Discriminator)

In [57]:
latent_dim = 8
G = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
D = nn.Sequential(nn.Linear(1, 32), nn.LeakyReLU(0.2), nn.Linear(32, 1), nn.Sigmoid()).to(device)
opt_G = torch.optim.Adam(G.parameters(), lr=1e-3)
opt_D = torch.optim.Adam(D.parameters(), lr=1e-3)
bce = nn.BCELoss()

# Contoh sederhana: Generator belajar meniru data normal dengan mean sekitar 0.
for epoch in range(100):
    real = torch.randn(64, 1, device=device)
    noise = torch.randn(64, latent_dim, device=device)
    fake = G(noise).detach()

    opt_D.zero_grad()
    loss_D = bce(D(real), torch.ones(64, 1, device=device)) + bce(D(fake), torch.zeros(64, 1, device=device))
    loss_D.backward()
    opt_D.step()

    opt_G.zero_grad()
    fake = G(torch.randn(64, latent_dim, device=device))
    loss_G = bce(D(fake), torch.ones(64, 1, device=device))
    loss_G.backward()
    opt_G.step()

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1}/100 - D loss: {loss_D.item():.4f} - G loss: {loss_G.item():.4f}")

Epoch 25/100 - D loss: 1.3775 - G loss: 0.7185
Epoch 50/100 - D loss: 1.3405 - G loss: 0.7620
Epoch 75/100 - D loss: 1.2830 - G loss: 0.8093
Epoch 100/100 - D loss: 1.3964 - G loss: 0.6703


## 4. TensorFlow / Keras 

TensorFlow is optional. The examples below are skipped if the package is not installed.

In [58]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_AVAILABLE = True
    print("TensorFlow:", tf.__version__)
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow belum terpasang. Jalankan: pip install tensorflow")

if TF_AVAILABLE:
    tf.random.set_seed(RANDOM_STATE)
    rng_tf = np.random.default_rng(RANDOM_STATE)

    # Tabular data
    X_tf = rng_tf.normal(size=(256, 12)).astype("float32")
    y_tf = (X_tf[:, :3].sum(axis=1) > 0).astype("float32")
    y_tf_reg = (2 * X_tf[:, 0] - X_tf[:, 1] + 0.1 * rng_tf.normal(size=256)).astype("float32")
    X_tf_train, X_tf_test, y_tf_train, y_tf_test = train_test_split(
        X_tf, y_tf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_tf
    )
    X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
        X_tf, y_tf_reg, test_size=0.2, random_state=RANDOM_STATE
    )

    # Image data: horizontal and vertical lines with a little noise
    X_img_tf = np.zeros((256, 28, 28, 1), dtype="float32")
    y_img_tf = (np.arange(256) % 2).astype("int32")
    for i in range(len(X_img_tf)):
        if y_img_tf[i] == 0:
            X_img_tf[i, 12:16, :, 0] = 1.0
        else:
            X_img_tf[i, :, 12:16, 0] = 1.0
    X_img_tf += rng_tf.normal(0, 0.15, X_img_tf.shape).astype("float32")
    X_img_train, X_img_test, y_img_train, y_img_test = train_test_split(
        X_img_tf, y_img_tf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_img_tf
    )

    # Sequence data
    X_seq_tf = rng_tf.normal(size=(256, 20, 8)).astype("float32")
    y_seq_tf = (X_seq_tf.mean(axis=(1, 2)) > 0).astype("float32")
    X_seq_train, X_seq_test, y_seq_train, y_seq_test = train_test_split(
        X_seq_tf, y_seq_tf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_seq_tf
    )

TensorFlow: 2.20.0


### 01. Sequential MLP untuk klasifikasi tabular

In [59]:
if TF_AVAILABLE:
    model = keras.Sequential([
        layers.Input(shape=(12,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(
        X_tf_train, y_tf_train, epochs=5, batch_size=32,
        validation_split=0.2, verbose=0
    )
    print("Test:", model.evaluate(X_tf_test, y_tf_test, verbose=0))

Test: [0.5515641570091248, 0.75]


### 02. Functional API MLP (multi-input / fleksibel)

In [60]:
if TF_AVAILABLE:
    inputs = keras.Input(shape=(12,), name="features")
    x = layers.Dense(64, activation="relu")(inputs)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="functional_mlp")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_tf_train, y_tf_train, epochs=5, batch_size=32, validation_split=0.2, verbose=0)
    print("Test:", model.evaluate(X_tf_test, y_tf_test, verbose=0))

Test: [0.5589969158172607, 0.7307692170143127]


### 03. MLP untuk regresi

In [61]:
if TF_AVAILABLE:
    model = keras.Sequential([
        layers.Input(shape=(12,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    model.fit(X_reg_train, y_reg_train, epochs=5, batch_size=32, validation_split=0.2, verbose=0)
    print("Test:", model.evaluate(X_reg_test, y_reg_test, verbose=0))

Test: [2.6727235317230225, 1.332260251045227]


### 04. CNN untuk klasifikasi gambar

In [62]:
if TF_AVAILABLE:
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(16, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(2, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(X_img_train, y_img_train, epochs=4, batch_size=32, validation_split=0.2, verbose=0)
    print("Test:", model.evaluate(X_img_test, y_img_test, verbose=0))

Test: [0.0001201481863972731, 1.0]


### 05. SimpleRNN untuk sequence classification

In [63]:
if TF_AVAILABLE:
    model = keras.Sequential([
        layers.Input(shape=(20, 8)),
        layers.SimpleRNN(32),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_seq_train, y_seq_train, epochs=4, batch_size=32, validation_split=0.2, verbose=0)
    print("Test:", model.evaluate(X_seq_test, y_seq_test, verbose=0))

Test: [0.8238644003868103, 0.36538460850715637]


### 06. LSTM untuk sequence classification

In [64]:
if TF_AVAILABLE:
    model = keras.Sequential([
        layers.Input(shape=(20, 8)),
        layers.LSTM(32),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_seq_train, y_seq_train, epochs=4, batch_size=32, validation_split=0.2, verbose=0)
    print("Test:", model.evaluate(X_seq_test, y_seq_test, verbose=0))

Test: [0.6253543496131897, 0.7692307829856873]


### 07. GRU untuk sequence classification

In [65]:
if TF_AVAILABLE:
    model = keras.Sequential([
        layers.Input(shape=(20, 8)),
        layers.GRU(32),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_seq_train, y_seq_train, epochs=4, batch_size=32, validation_split=0.2, verbose=0)
    print("Test:", model.evaluate(X_seq_test, y_seq_test, verbose=0))

Test: [0.6364728212356567, 0.6730769276618958]


### 08. Bidirectional LSTM

In [66]:
if TF_AVAILABLE:
    model = keras.Sequential([
        layers.Input(shape=(20, 8)),
        layers.Bidirectional(layers.LSTM(32)),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_seq_train, y_seq_train, epochs=4, batch_size=32, validation_split=0.2, verbose=0)
    print("Test:", model.evaluate(X_seq_test, y_seq_test, verbose=0))

Test: [0.5923134088516235, 0.8269230723381042]


### 09. Autoencoder for feature learning

In [67]:
if TF_AVAILABLE:
    inputs = keras.Input(shape=(12,))
    encoded = layers.Dense(8, activation="relu")(inputs)
    latent = layers.Dense(3, activation="relu", name="latent_space")(encoded)
    decoded = layers.Dense(8, activation="relu")(latent)
    outputs = layers.Dense(12)(decoded)
    autoencoder = keras.Model(inputs, outputs)
    autoencoder.compile(optimizer="adam", loss="mse")
    autoencoder.fit(
        X_tf_train, X_tf_train, epochs=5, batch_size=32,
        validation_data=(X_tf_test, X_tf_test), verbose=0
    )
    encoder = keras.Model(inputs, latent)
    X_latent = encoder.predict(X_tf_test[:5], verbose=0)
    print("Latent shape:", X_latent.shape)

Latent shape: (5, 3)


### 10. Transformer Encoder for Sequens Classification

In [68]:
if TF_AVAILABLE:
    def transformer_encoder(inputs, positions, head_size=16, num_heads=2, ff_dim=32, dropout=0.1):
        # Positional embedding memberi informasi urutan pada input.
        x = inputs + positions
        y = layers.LayerNormalization(epsilon=1e-6)(x)
        y = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size, dropout=dropout)(y, y)
        y = layers.Dropout(dropout)(y)
        res = x + y
        y = layers.LayerNormalization(epsilon=1e-6)(res)
        y = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(y)
        y = layers.Dropout(dropout)(y)
        y = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(y)
        return y + res

    inputs = keras.Input(shape=(20, 8))
    position_ids = tf.range(start=0, limit=20, delta=1)
    position_embedding = layers.Embedding(input_dim=20, output_dim=8)(position_ids)
    x = transformer_encoder(inputs, position_embedding)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="transformer_encoder_classifier")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_seq_train, y_seq_train, epochs=4, batch_size=32, validation_split=0.2, verbose=0)
    print("Test:", model.evaluate(X_seq_test, y_seq_test, verbose=0))

Test: [0.6840866804122925, 0.6346153616905212]


## 5. Guide to model selection and evaluation


| Problem                  | Recommended Baseline Models                                | Common Evaluation Metrics                            |
| ------------------------ | ---------------------------------------------------------- | ---------------------------------------------------- |
| Tabular Regression       | Linear Regression, Ridge, Random Forest, Gradient Boosting | MAE, RMSE, R²                                        |
| Tabular Classification   | Logistic Regression, Random Forest, SVM, Gradient Boosting | F1-score, Precision, Recall, ROC-AUC                 |
| Clustering               | K-Means, DBSCAN, Agglomerative Clustering                  | Silhouette Score; domain validation                  |
| Dimensionality Reduction | PCA, TruncatedSVD                                          | Explained Variance / Downstream Score                |
| Anomaly Detection        | Isolation Forest, LOF, One-Class SVM                       | Precision / Recall when anomaly labels are available |
| Image Data               | CNN / Pretrained Vision Models                             | Accuracy, F1-score, Confusion Matrix                 |
| Sequential Data          | LSTM, GRU, Transformer                                     | Metrics appropriate to the prediction target         |

Important Notes:

No single algorithm is universally the best choice for every problem. Consider the following practices when developing machine learning models:

1. Split your dataset into training, validation, and test sets.
2. Perform preprocessing using only the training data. Use `Pipeline` to prevent data leakage.
3. Establish baseline models before experimenting with more complex algorithms.
4. Perform hyperparameter tuning using validation data or cross-validation (CV).
5. Report final evaluation metrics using a separate test set that has not been used for model selection.
6. For imbalanced datasets, do not rely solely on accuracy. Use metrics such as Precision, Recall, F1-score, and ROC-AUC when appropriate.

For the `MLWorkbench` section:

1. `compare()` compares candidate models with cross-validation on the training portion.
2. Choose a model using the cross-validation results.
3. Use `run()` once for the final test-set evaluation of the selected model.

Do not keep changing the model based on its test-set score.

## 6. Practical Use

This section uses a sample dataset so that you can try it out immediately. For your own dataset, replace `demo_df` with the DataFrame obtained from `pd.read_csv()` in the final cell.

In [69]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score, silhouette_score
)
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression,
    SGDClassifier, Perceptron, PassiveAggressiveClassifier
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier,
    ExtraTreesRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor,
    AdaBoostClassifier, BaggingClassifier, VotingClassifier, StackingClassifier,
    IsolationForest
)
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.cluster import (
    KMeans, MiniBatchKMeans, DBSCAN, AgglomerativeClustering, SpectralClustering
)
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA, TruncatedSVD

In [70]:
class MLWorkbench:

    CATALOG = {
        1: ("Linear Regression", "regression"),
        2: ("Ridge Regression", "regression"),
        3: ("Lasso Regression", "regression"),
        4: ("Elastic Net", "regression"),
        5: ("Logistic Regression", "classification"),
        6: ("SGD Classifier", "classification"),
        7: ("Perceptron", "classification"),
        8: ("Passive Aggressive Classifier", "classification"),
        9: ("KNN Classifier", "classification"),
        10: ("KNN Regressor", "regression"),
        11: ("SVC", "classification"),
        12: ("SVR", "regression"),
        13: ("Decision Tree Classifier", "classification"),
        14: ("Decision Tree Regressor", "regression"),
        15: ("Random Forest Classifier", "classification"),
        16: ("Random Forest Regressor", "regression"),
        17: ("Extra Trees Classifier", "classification"),
        18: ("Extra Trees Regressor", "regression"),
        19: ("Gradient Boosting Classifier", "classification"),
        20: ("Gradient Boosting Regressor", "regression"),
        21: ("Histogram Gradient Boosting Classifier", "classification"),
        22: ("Histogram Gradient Boosting Regressor", "regression"),
        23: ("AdaBoost Classifier", "classification"),
        24: ("Bagging Classifier", "classification"),
        25: ("Voting Classifier", "classification"),
        26: ("Stacking Classifier", "classification"),
        27: ("Gaussian Naive Bayes", "classification"),
        28: ("Multinomial Naive Bayes", "classification"),
        29: ("Bernoulli Naive Bayes", "classification"),
        30: ("Linear Discriminant Analysis", "classification"),
        31: ("Quadratic Discriminant Analysis", "classification"),
        32: ("K-Means", "clustering"),
        33: ("Mini-Batch K-Means", "clustering"),
        34: ("DBSCAN", "clustering"),
        35: ("Agglomerative Clustering", "clustering"),
        36: ("Spectral Clustering", "clustering"),
        37: ("Gaussian Mixture Model", "clustering"),
        38: ("PCA", "dimensionality_reduction"),
        39: ("Truncated SVD", "dimensionality_reduction"),
        40: ("Isolation Forest", "anomaly_detection")
    }

    RECOMMENDATIONS = {
        "classification": [5, 15, 19, 11],
        "regression": [1, 2, 16, 20],
        "clustering": [32, 34, 35],
        "dimensionality_reduction": [38, 39],
        "anomaly_detection": [40]
    }

    def __init__(
        self,
        data,
        target=None,
        test_size=0.2,
        random_state=RANDOM_STATE
    ):

        if not isinstance(data, pd.DataFrame):
            raise TypeError("Dataset must be a Pandas DataFrame.")

        if data.empty:
            raise ValueError("Dataset is empty.")

        if not 0 < test_size < 1:
            raise ValueError("test_size must be between 0 and 1.")

        if target is not None and target not in data.columns:
            raise ValueError(f"Target '{target}' does not exist.")

        self.data = data.copy()
        self.target = target
        self.test_size = test_size
        self.random_state = random_state

        self.last_model = None
        self.last_preprocessor = None
        self.last_result = None
        self.last_task = None

        print("Dataset successfully registered.")
        print("Shape:", self.data.shape)

# 1. DISPLAY DATASET

    def inspect(self):

        print("\nDATASET OVERVIEW")
        print("=" * 55)

        print("Shape:", self.data.shape)

        print("\nColumns:")
        print(self.data.columns.tolist())

        print("\nData Types:")
        print(self.data.dtypes)

        print("\nMissing Values:")
        print(self.data.isnull().sum())

        print("\nFirst Five Rows:")
        print(self.data.head())

# 2. DISPLAY ALGORITHMS

    def show_algorithms(self, task=None):

        print("\nAVAILABLE ALGORITHMS")
        print("=" * 70)

        for number, (name, category) in self.CATALOG.items():

            if task is None or category == task:

                print(
                    f"{number:02d}. "
                    f"{name:<45} | {category}"
                )

        print("=" * 70)

# 3. RECOMMENDATION
    
    def recommend(self, task):

        if task not in self.RECOMMENDATIONS:
            raise ValueError("Unknown machine learning task.")

        print(f"\nRECOMMENDED STARTING MODELS: {task.upper()}")
        print("-" * 55)

        for number in self.RECOMMENDATIONS[task]:

            name, category = self.CATALOG[number]

            print(f"{number:02d}. {name}")

        print("\nThese are baseline candidates, not guaranteed winners.")

 # 4. AUTOMATIC PREPROCESSING

    def _preprocess(self, X, algorithm_number):

        X = X.dropna(axis=1, how="all")

        if X.shape[1] == 0:
            raise ValueError("No usable features were found.")

        numerical = X.select_dtypes(
            include=["number", "bool"]
        ).columns.tolist()

        categorical = X.select_dtypes(
            include=["object", "category", "string"]
        ).columns.tolist()

        unsupported = [
            col for col in X.columns
            if col not in numerical and col not in categorical
        ]

        if unsupported:
            raise ValueError(
                f"Unsupported columns: {unsupported}. "
                "Convert dates or special data types first."
            )

        transformers = []

        if numerical:

            if algorithm_number == 28:

                num_pipeline = Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", MinMaxScaler(clip=True))
                ])

            elif algorithm_number == 29:

                num_pipeline = Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("binarizer", KBinsDiscretizer(
                        n_bins=2,
                        encode="onehot-dense",
                        strategy="quantile"
                    ))
                ])

            else:

                num_pipeline = Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ])

            transformers.append(
                ("numerical", num_pipeline, numerical)
            )

        if categorical:

            try:
                encoder = OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            except TypeError:
                encoder = OneHotEncoder(
                    handle_unknown="ignore",
                    sparse=False
                )

            cat_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", encoder)
            ])

            transformers.append(
                ("categorical", cat_pipeline, categorical)
            )

        if not transformers:
            raise ValueError("No valid features found.")

        return ColumnTransformer(
            transformers=transformers,
            remainder="drop"
        )

# 5. MODEL CONSTRUCTION

    def _build_model(
        self,
        number,
        n_train,
        n_samples,
        n_features,
        n_clusters,
        stacking_cv=5
    ):

        models = {

            1: LinearRegression(),

            2: Ridge(alpha=1.0),

            3: Lasso(alpha=0.1, max_iter=10000),

            4: ElasticNet(
                alpha=0.1,
                l1_ratio=0.5,
                max_iter=10000
            ),

            5: LogisticRegression(max_iter=2000),

            6: SGDClassifier(
                max_iter=2000,
                random_state=RANDOM_STATE
            ),

            7: Perceptron(
                max_iter=2000,
                random_state=RANDOM_STATE
            ),

            8: PassiveAggressiveClassifier(
                max_iter=2000,
                random_state=RANDOM_STATE
            ),

            9: KNeighborsClassifier(
                n_neighbors=min(5, n_train)
            ),

            10: KNeighborsRegressor(
                n_neighbors=min(5, n_train)
            ),

            11: SVC(),

            12: SVR(),

            13: DecisionTreeClassifier(random_state=RANDOM_STATE),

            14: DecisionTreeRegressor(random_state=RANDOM_STATE),

            15: RandomForestClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),

            16: RandomForestRegressor(
                n_estimators=200,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),

            17: ExtraTreesClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),

            18: ExtraTreesRegressor(
                n_estimators=200,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),

            19: GradientBoostingClassifier(random_state=RANDOM_STATE),

            20: GradientBoostingRegressor(random_state=RANDOM_STATE),

            21: HistGradientBoostingClassifier(random_state=RANDOM_STATE),

            22: HistGradientBoostingRegressor(random_state=RANDOM_STATE),

            23: AdaBoostClassifier(random_state=RANDOM_STATE),

            24: BaggingClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),

            25: VotingClassifier(
                estimators=[
                    ("lr", LogisticRegression(max_iter=2000)),
                    ("rf", RandomForestClassifier(
                        n_estimators=100,
                        random_state=RANDOM_STATE
                    )),
                    ("svc", SVC())
                ],
                voting="hard"
            ),

            26: StackingClassifier(
                estimators=[
                    ("lr", LogisticRegression(max_iter=2000)),
                    ("rf", RandomForestClassifier(
                        n_estimators=100,
                        random_state=RANDOM_STATE
                    )),
                    ("svc", SVC())
                ],
                final_estimator=LogisticRegression(max_iter=2000),
                cv=stacking_cv
            ),

            27: GaussianNB(),

            28: MultinomialNB(),

            29: BernoulliNB(),

            30: LinearDiscriminantAnalysis(),

            31: QuadraticDiscriminantAnalysis()
        }

        if number in models:
            return models[number]

        k = int(n_clusters)

        if k < 2:
            raise ValueError("Number of clusters must be at least 2.")

        if k > n_samples:
            raise ValueError("Number of clusters exceeds sample count.")

        if number == 32:
            return KMeans(
                n_clusters=k,
                n_init=10,
                random_state=RANDOM_STATE
            )

        if number == 33:
            return MiniBatchKMeans(
                n_clusters=k,
                n_init=10,
                random_state=RANDOM_STATE
            )

        if number == 34:
            return DBSCAN(
                eps=0.5,
                min_samples=5
            )

        if number == 35:
            return AgglomerativeClustering(n_clusters=k)

        if number == 36:
            return SpectralClustering(
                n_clusters=k,
                n_neighbors=min(10, n_samples - 1),
                random_state=RANDOM_STATE
            )

        if number == 37:
            return GaussianMixture(
                n_components=k,
                random_state=RANDOM_STATE
            )

        if number == 38:
            return PCA(
                n_components=min(2, n_samples, n_features)
            )

        if number == 39:
            return TruncatedSVD(
                n_components=min(2, n_features),
                random_state=RANDOM_STATE
            )

        if number == 40:
            return IsolationForest(
                contamination="auto",
                random_state=RANDOM_STATE
            )

        raise ValueError("Invalid algorithm number.")

# 6. MAIN EXECUTION

    def run(
        self,
        algorithm_number=None,
        target=None,
        n_clusters=3,
        model_params=None
    ):

        # Interactive algorithm selection

        if algorithm_number is None:

            self.show_algorithms()

            algorithm_number = input(
                "\nSelect algorithm number (1-40): "
            )

            try:
                algorithm_number = int(algorithm_number)
            except ValueError:
                raise ValueError("Algorithm ID must be an integer.")

        if algorithm_number not in self.CATALOG:
            raise ValueError("Algorithm ID must be between 1 and 40.")

        name, task = self.CATALOG[algorithm_number]

        target = target if target is not None else self.target

        print("\nMODEL CONFIGURATION")
        print("=" * 55)
        print("Algorithm:", algorithm_number)
        print("Model:", name)
        print("Task:", task)

        self.last_task = task

# SUPERVISED LEARNING

        if task in ["classification", "regression"]:

            if target is None:
                target = input("Enter target column name: ").strip()

            if target not in self.data.columns:
                raise ValueError(f"Target '{target}' not found.")

            X = self.data.drop(columns=[target])
            y = self.data[target]

            valid = y.notna()

            X = X.loc[valid]
            y = y.loc[valid]

            X = X.dropna(axis=1, how="all")

            if X.shape[1] == 0:
                raise ValueError("No usable feature columns.")

            if task == "classification":

                if y.nunique() < 2:
                    raise ValueError("Classification needs at least two classes.")

            elif not pd.api.types.is_numeric_dtype(y):
                raise ValueError("Regression target must be numerical.")

            # Stratification when feasible

            stratify = None

            if task == "classification":

                n_classes = y.nunique()
                n_test = int(np.ceil(len(y) * self.test_size))
                n_train = len(y) - n_test

                if (
                    y.value_counts().min() >= 2
                    and n_test >= n_classes
                    and n_train >= n_classes
                ):
                    stratify = y

            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=self.test_size,
                random_state=self.random_state,
                stratify=stratify
            )

            if len(X_train) < 1 or len(X_test) < 1:
                raise ValueError("Insufficient observations for splitting.")

            stacking_cv = 5

            if algorithm_number == 26:

                min_class_count = y_train.value_counts().min()

                if min_class_count < 2:
                    raise ValueError(
                        "Stacking requires at least two training "
                        "observations per class."
                    )

                stacking_cv = min(5, int(min_class_count))

            preprocessor = self._preprocess(
                X_train,
                algorithm_number
            )

            model = self._build_model(
                algorithm_number,
                n_train=len(X_train),
                n_samples=len(X_train),
                n_features=X_train.shape[1],
                n_clusters=n_clusters,
                stacking_cv=stacking_cv
            )

            if model_params:
                model.set_params(**model_params)

            pipeline = Pipeline([
                ("preprocessing", preprocessor),
                ("model", model)
            ])

            print("\nTraining...")

            pipeline.fit(X_train, y_train)

            predictions = pipeline.predict(X_test)

            print("\nEVALUATION RESULTS")
            print("=" * 55)

            if task == "classification":

                metrics = {
                    "Accuracy": accuracy_score(y_test, predictions),

                    "Precision": precision_score(
                        y_test,
                        predictions,
                        average="macro",
                        zero_division=0
                    ),

                    "Recall": recall_score(
                        y_test,
                        predictions,
                        average="macro",
                        zero_division=0
                    ),

                    "F1 Score": f1_score(
                        y_test,
                        predictions,
                        average="macro",
                        zero_division=0
                    )
                }

                print(pd.Series(metrics))

                print("\nClassification Report:")
                print(classification_report(
                    y_test,
                    predictions,
                    zero_division=0
                ))

                print("\nConfusion Matrix:")
                print(confusion_matrix(y_test, predictions))

            else:

                metrics = {
                    "MAE": mean_absolute_error(y_test, predictions),

                    "RMSE": np.sqrt(
                        mean_squared_error(y_test, predictions)
                    ),

                    "R2": r2_score(y_test, predictions)
                }

                print(pd.Series(metrics))

            result = {
                "model": pipeline,
                "predictions": predictions,
                "metrics": metrics,
                "X_test": X_test,
                "y_test": y_test
            }

            self.last_model = pipeline
            self.last_preprocessor = None
            self.last_result = result
            self.target = target

            return result

# UNSUPERVISED LEARNING
        
        X = (
            self.data.drop(columns=[target])
            if target is not None
            else self.data.copy()
        )

        X = X.dropna(axis=1, how="all")

        if len(X) < 2:
            raise ValueError("At least two observations are required.")

        preprocessor = self._preprocess(
            X,
            algorithm_number
        )

        X_processed = preprocessor.fit_transform(X)

        model = self._build_model(
            algorithm_number,
            n_train=len(X_processed),
            n_samples=len(X_processed),
            n_features=X_processed.shape[1],
            n_clusters=n_clusters
        )

        if model_params:
            model.set_params(**model_params)

        print("\nExecuting model...")

        # Dimensionality reduction

        if task == "dimensionality_reduction":

            transformed = model.fit_transform(X_processed)

            output = pd.DataFrame(
                transformed,
                columns=[
                    f"Component_{i+1}"
                    for i in range(transformed.shape[1])
                ],
                index=X.index
            )

            metrics = {}

            if hasattr(model, "explained_variance_ratio_"):

                metrics["Explained Variance Ratio"] = (
                    model.explained_variance_ratio_
                )

                print("\nExplained Variance:")
                print(model.explained_variance_ratio_)

            result = {
                "model": model,
                "transformed_data": output,
                "metrics": metrics
            }

        else:

            labels = model.fit_predict(X_processed)

            output = X.copy()
            output["Model_Output"] = labels

            metrics = {}

            if task == "clustering":

                valid = labels != -1
                valid_labels = labels[valid]
                valid_data = X_processed[valid]

                unique_labels = np.unique(valid_labels)

                if (
                    1 < len(unique_labels) < len(valid_labels)
                ):

                    try:

                        metrics["Silhouette Score"] = silhouette_score(
                            valid_data,
                            valid_labels,
                            sample_size=min(2000, len(valid_labels)),
                            random_state=self.random_state
                        )

                    except ValueError:
                        pass

            elif task == "anomaly_detection":

                metrics["Normal Observations"] = int(
                    np.sum(labels == 1)
                )

                metrics["Anomalies"] = int(
                    np.sum(labels == -1)
                )

            print("\nModel Output:")
            print(output.head())

            print("\nEvaluation:")
            print(metrics)

            result = {
                "model": model,
                "labels": labels,
                "result": output,
                "metrics": metrics
            }

        self.last_model = model
        self.last_preprocessor = preprocessor
        self.last_result = result

        return result

# 7. PREDICT NEW DATA

    def predict(self, new_data):

        if self.last_model is None:
            raise RuntimeError("Train a model first.")

        if not isinstance(new_data, pd.DataFrame):
            raise TypeError("Input must be a Pandas DataFrame.")

        if self.last_task not in ["classification", "regression"]:
            raise ValueError(
                "Prediction through this function is available "
                "for supervised models only."
            )

        predictions = self.last_model.predict(new_data)

        return predictions

#8. COMPARE MULTIPLE MODELS

    def compare(self, algorithm_numbers, target=None, cv=5):

        target = target if target is not None else self.target

        if target is None:
            raise ValueError("Specify the target column.")

        if target not in self.data.columns:
            raise ValueError(f"Target '{target}' not found.")

        if not algorithm_numbers:
            raise ValueError("Provide at least one algorithm number.")

        tasks = [self.CATALOG[number][1] for number in algorithm_numbers]

        if len(set(tasks)) != 1:
            raise ValueError("Compare one task at a time: classification or regression.")

        task = tasks[0]
        if task not in ["classification", "regression"]:
            raise ValueError("compare() is for classification or regression models.")

        X = self.data.drop(columns=[target])
        y = self.data[target]
        valid = y.notna()
        X, y = X.loc[valid], y.loc[valid]
        X = X.dropna(axis=1, how="all")

        stratify = y if task == "classification" and y.value_counts().min() >= 2 else None
        X_train, _, y_train, _ = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state,
            stratify=stratify
        )

        if task == "classification":
            min_count = int(y_train.value_counts().min())
            n_splits = min(int(cv), min_count)
            if n_splits < 2:
                raise ValueError("Not enough examples per class for cross-validation.")
            splitter = StratifiedKFold(
                n_splits=n_splits, shuffle=True, random_state=self.random_state
            )
            scoring = {
                "Accuracy": "accuracy",
                "Precision": "precision_macro",
                "Recall": "recall_macro",
                "F1": "f1_macro"
            }
        else:
            n_splits = min(int(cv), len(X_train))
            if n_splits < 2:
                raise ValueError("Not enough rows for cross-validation.")
            splitter = KFold(
                n_splits=n_splits, shuffle=True, random_state=self.random_state
            )
            scoring = {
                "MAE": "neg_mean_absolute_error",
                "RMSE": "neg_root_mean_squared_error",
                "R2": "r2"
            }

        results = []

        for number in algorithm_numbers:
            name, _ = self.CATALOG[number]
            try:
                stacking_cv = 5
                if number == 26 and task == "classification":
                    # Inner CV for Stacking must fit inside each outer CV fold.
                    fold_train_count = min_count - int(np.ceil(min_count / n_splits))
                    stacking_cv = max(2, min(5, fold_train_count))

                preprocessor = self._preprocess(X_train, number)
                model = self._build_model(
                    number,
                    n_train=len(X_train),
                    n_samples=len(X_train),
                    n_features=X_train.shape[1],
                    n_clusters=3,
                    stacking_cv=stacking_cv
                )
                pipeline = Pipeline([
                    ("preprocessing", preprocessor),
                    ("model", model)
                ])

                scores = cross_validate(
                    pipeline, X_train, y_train,
                    cv=splitter, scoring=scoring,
                    return_train_score=False, error_score="raise"
                )

                row = {"ID": number, "Model": name, "CV Folds": n_splits}
                for metric in scoring:
                    values = scores[f"test_{metric}"]
                    if metric in ["MAE", "RMSE"]:
                        values = -values
                    row[f"{metric} Mean"] = float(np.mean(values))
                    row[f"{metric} Std"] = float(np.std(values))
                results.append(row)

            except Exception as error:
                results.append({
                    "ID": number,
                    "Model": name,
                    "CV Folds": n_splits,
                    "Error": str(error)
                })

        table = pd.DataFrame(results)
        sort_column = "F1 Mean" if task == "classification" else "RMSE Mean"
        if sort_column in table.columns:
            table = table.sort_values(sort_column, ascending=(task == "regression"), na_position="last")
        return table.reset_index(drop=True)


## Lets use

In [ ]:
# To use your own dataset, replace the example with:
# df = pd.read_csv("your_dataset.csv")
# target_column = "your_target_column"

ml = MLWorkbench(data=df, target=target_column)
ml.inspect()
ml.show_algorithms(task="classification")

In [ ]:
# Compare several models using cross-validation.
comparison = ml.compare(algorithm_numbers=[5, 9, 11, 13, 15, 19], cv=5)
print(comparison.to_string(index=False))

In [ ]:
# After selecting a model based on cross-validation, perform the final evaluation on the test set.
result = ml.run(algorithm_number=15)
print(result["metrics"])

In [ ]:
# Example prediction on new data. The columns must match the training features.
new_data = df.drop(columns="target").head()
predictions = ml.predict(new_data)
print("Predictions:", predictions)